In [9]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
import csv

def get_movie_data(imdb_url):
    # Send a GET request to the IMDb page
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(imdb_url, headers=headers)

    # Check if the request was successful
    if response.status_code != 200:
        raise Exception(f"Failed to retrieve IMDb page. Status code: {response.status_code}")

    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.content, 'html.parser')

    # Find the script tag containing the JSON data
    script = soup.find('script', {'type': "application/ld+json"})
    if not script:
        raise Exception("No JSON data found in the page.")

    data = json.loads(script.text)
    print(data)
    # Flatten the JSON data
    flattened_data = {
        "context": data.get("@context", ""),
        "type": data.get("@type", ""),
        "url": data.get("url", ""),
        "name": data.get("name", ""),
        "image": data.get("image", ""),
        "contentRating": data.get("contentRating", ""),
        "genre": ", ".join(data.get("genre", [])),  # Combine genres into a single string
        "datePublished": data.get("datePublished", ""),
        "actors": ", ".join([actor.get("name", "") for actor in data.get("actor", [])]),  # Combine actor names
        "directors": ", ".join([director.get("name", "") for director in data.get("director", [])]),  # Combine director names
        "creator": data.get("creator", [{}])[0].get("url", "") if data.get("creator") else ""  # Extract creator URL
    }
    
    # Convert to DataFrame
    df = pd.DataFrame([flattened_data])

    return df

def main():
    topdf = pd.read_csv("./top_tv_shows_full_data.csv")
    with open('top_tv_shows_complete.csv', 'w', newline='', encoding='utf-8') as csvfile:
        # Create a CSV writer
        csvwriter = csv.writer(csvfile)
        # Write the header row (fields)
        csvwriter.writerow(['Context', 'Type', 'URL', 'Name', 'Poster', 'Content Rating', 'Genre', 
                            'Date', 'Actor','Director', 'Creator'])
        for i in range(0, 249):
            print(f"Processing movie {i+1} {topdf['Show Name']} of 250")
            imdb_url = topdf['URL'][i]
            finalurl = imdb_url.replace("https://www.imdb.com", "", 1)
            try:
                movie_data = get_movie_data(finalurl)
                csvwriter.writerow([movie_data['context'][0], movie_data['type'][0], 
                                    movie_data['url'][0], movie_data['name'][0], 
                                    movie_data['image'][0], movie_data['contentRating'][0], 
                                    movie_data['genre'][0], movie_data['datePublished'][0], 
                                    movie_data['actors'][0], movie_data['directors'][0], movie_data['creator'][0]])   


            except Exception as e:
                print(f"Error: {e}")
    
    # Example IMDb URL for "The Shawshank Redemption"
    #imdb_url = f"https://www.imdb.com/title/tt{random.randint(1, 9999999):07d}/"
    
    #try:
        # Get movie data
    #    movie_data = get_movie_data(imdb_url)
    #    print(movie_data)
    #except Exception as e:
    #    print(f"Error: {e}")

if __name__ == "__main__":
    main()

Processing movie 1 0                    Breaking Bad
1                 Planet Earth II
2                    Planet Earth
3                Band of Brothers
4                       Chernobyl
                  ...            
245                   Letterkenny
246                       Gintama
247              Foyle&apos;s War
248    The Great British Bake Off
249            Miseuteo Shunshain
Name: Show Name, Length: 250, dtype: object of 250
{'@context': 'https://schema.org', '@type': 'TVSeries', 'url': 'https://www.imdb.com/title/tt0903747/', 'name': 'Breaking Bad', 'image': 'https://m.media-amazon.com/images/M/MV5BMzU5ZGYzNmQtMTdhYy00OGRiLTg0NmQtYjVjNzliZTg1ZGE4XkEyXkFqcGc@._V1_.jpg', 'description': 'A chemistry teacher diagnosed with inoperable lung cancer turns to manufacturing and selling methamphetamine with a former student to secure his family&apos;s future.', 'review': {'@type': 'Review', 'itemReviewed': {'@type': 'TVSeries', 'url': 'https://www.imdb.com/title/tt0903747/'}, 'aut